In [0]:

%run ./00-config

### Event Hub Producer (Message Ingestion Engine)
* Authenticates to Azure Key Vault scope to retrieve the Event Hub Connection String securely.
* Generates synthetic real-time event payloads and sends them in batches to `natalkamartinuk55-stream-hub`.

In [0]:
import json
import uuid
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData

conn_str = dbutils.secrets.get(scope=scope_name, key=secret_key)

producer = EventHubProducerClient.from_connection_string(
    conn_str=conn_str, 
    eventhub_name=eventhub_name
)

events_data = []
categories = ["Music", "Gaming", "News", "Education", "Entertainment"]

for i in range(25):
    event_payload = {
        "event_id": str(uuid.uuid4()),
        "video_id": f"VID_{1000 + i}",
        "user_id": f"USER_{500 + (i % 10)}",
        "category": categories[i % len(categories)],
        "watch_duration_sec": 45 + (i * 7),
        "event_timestamp": datetime.utcnow().isoformat()
    }
    events_data.append(event_payload)

event_batch = producer.create_batch()
for event in events_data:
    event_batch.add(EventData(json.dumps(event)))

with producer:
    producer.send_batch(event_batch)